# Mão na Roda — Fine-tuning do JabuticaBERT
Diagnóstico Preliminar de Falhas Automotivas Baseado em IA

Notebook equivalente aos do BERTimbau, BERTabaporu e DeBERTinha, trocando o
modelo base para o **JabuticaBERT** (`amadeusai/JabuticaBERT-XSmall`).

**Sobre o modelo:** publicado por Porto, Gomes, Bender, Corrêa, Freitas,
Cruz e Amadeus no PROPOR 2026 ("JabuticaBERT: Modern Portuguese Encoders
from Scratch with RTD and Long-Context Training"). É um modelo DeBERTa
treinado **do zero** (não é um fine-tuning de um modelo em inglês) com
Replaced Token Detection (RTD), no corpus **Jabuticaba** — o maior corpus
de português já reunido para esse fim, com 669GB e mais de 139 bilhões de
tokens (Amadeus et al., 2024). O artigo reporta que a versão *Large* do
JabuticaBERT iguala o desempenho do Albertina de 900M de parâmetros
usando uma fração dos parâmetros. Este notebook usa a versão **XSmall**,
mais leve, compatível com a GPU T4 gratuita do Colab e com porte
comparável ao DeBERTinha (~40M de parâmetros), para manter a bateria de
modelos comparável em escala computacional.

**Por que esse experimento é diferente dos anteriores:** BERTimbau e
BERTabaporu são arquitetura BERT (WordPiece, MLM). DeBERTinha e
JabuticaBERT são ambos DeBERTa com RTD — mas foram pré-treinados em
corpora de proveniência e escala muito diferentes (DeBERTinha é adaptado
de um checkpoint em inglês via destilação; JabuticaBERT é treinado do
zero em português). Isso permite uma comparação mais fina: "arquitetura
igual, origem de pré-treino diferente" — complementando a comparação
"arquitetura diferente" que já foi feita entre BERTimbau e DeBERTinha.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de
execução → GPU (T4).

## Passo 1 — Setup e verificação do ambiente

Inclui a remoção preventiva do `torchvision`, mesma correção de bug de
ambiente aplicada nos notebooks do BERTabaporu e do DeBERTinha.

In [1]:
# ============================================================
# PASSO 1 — Setup e verificação do ambiente
# Projeto Mão na Roda — Fine-tuning do JabuticaBERT
# ============================================================

# --- 1.0 Corrige bug de ambiente conhecido (datasets x torchvision) ---
!pip uninstall -y torchvision -q

# --- 1.1 Instala as dependências ---
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl sentencepiece

# --- 1.2 Verifica se a GPU está disponível ---
import torch
print("Torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  ATENÇÃO: nenhuma GPU detectada. Vá em 'Ambiente de execução' > "
          "'Alterar tipo de ambiente de execução' e selecione GPU antes de continuar.")

# --- 1.3 Constantes do projeto ---
# Único ponto que muda em relação aos notebooks anteriores: o checkpoint.
MODEL_CHECKPOINT = "amadeusai/JabuticaBERT-XSmall"  # JabuticaBERT (Porto et al., PROPOR 2026)
SEED = 42
NUM_LABELS = 10  # as 10 classes de falha do projeto
MAX_LENGTH = 64  # mesmo valor usado nos notebooks anteriores, para comparação justa

LABEL_NAMES = [
    "Bateria/Sistema Elétrico",       # 0
    "Sistema de Freios",              # 1
    "Superaquecimento do Motor",      # 2
    "Suspensão/Amortecedores",        # 3
    "Sistema de Transmissão/Câmbio",  # 4
    "Vazamento de Óleo",              # 5
    "Sistema de Arrefecimento",       # 6
    "Pneu/Roda",                      # 7
    "Sistema de Injeção/Combustível", # 8
    "Sistema de Escapamento",         # 9
]

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print("Setup concluído. Modelo base:", MODEL_CHECKPOINT)

# ============================================================
# --- 1.4 Carregando os arquivos de split (via Google Drive) ---
# ============================================================
# Reutiliza os MESMOS splits dos notebooks anteriores — essencial para
# que a comparação entre os 4 modelos seja válida.

from google.colab import drive
drive.mount('/content/drive')

CAMINHO_BASE = '/content/drive/MyDrive/Agente Mecânico/notebooks/Finetuning - 3000 relatos/3 Splits - Finetuning/Splits'

import pandas as pd
import os

CONFIGS = ["80_10_10", "70_15_15", "60_20_20"]

splits = {}
for tag in CONFIGS:
    splits[tag] = {
        "treino": pd.read_csv(os.path.join(CAMINHO_BASE, f"split_treino_{tag}.csv")),
        "val":    pd.read_csv(os.path.join(CAMINHO_BASE, f"split_val_{tag}.csv")),
        "teste":  pd.read_csv(os.path.join(CAMINHO_BASE, f"split_teste_{tag}.csv")),
    }
    n_tr, n_val, n_te = len(splits[tag]["treino"]), len(splits[tag]["val"]), len(splits[tag]["teste"])
    print(f"{tag}: treino={n_tr} | val={n_val} | teste={n_te} | total={n_tr+n_val+n_te}")

assert "peso_treino" in splits["80_10_10"]["treino"].columns, \
    "Coluna peso_treino não encontrada — confira se subiu o CSV certo (do v15, não uma versão antiga)."
print("\nColuna peso_treino presente. Pronto para o Passo 2 (tokenização).")

Torch: 2.11.0+cu128
GPU disponível: True
GPU: Tesla T4

Setup concluído. Modelo base: amadeusai/JabuticaBERT-XSmall
Mounted at /content/drive
80_10_10: treino=2432 | val=305 | teste=304 | total=3041
70_15_15: treino=2171 | val=435 | teste=435 | total=3041
60_20_20: treino=1824 | val=608 | teste=609 | total=3041

Coluna peso_treino presente. Pronto para o Passo 2 (tokenização).


## Passo 2 — Tokenização

O tokenizer é carregado a partir do próprio `MODEL_CHECKPOINT`. O
JabuticaBERT, por ter sido treinado do zero em português (não adaptado
de um checkpoint em inglês), tem vocabulário nativo — vale conferir na
checagem de truncamento abaixo se isso resulta em menos tokens por
relato (vocabulário melhor ajustado ao português tende a "quebrar" menos
palavras em sub-tokens).

In [2]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch["relato"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def preparar_dataset_hf(df):
    # Converte um DataFrame pandas em um Dataset do Hugging Face,
    # já tokenizado e com as colunas 'labels' e 'weight' prontas para o Trainer.
    ds = Dataset.from_pandas(df[["relato", "falha_label", "peso_treino"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.rename_column("falha_label", "labels")
    ds = ds.rename_column("peso_treino", "weight")
    ds = ds.remove_columns(["relato"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "weight"])
    return ds

# Tokeniza as 3 configs de uma vez (usadas na comparação leve do Passo 4)
datasets_tokenizados = {}
for tag in CONFIGS:
    datasets_tokenizados[tag] = {
        "treino": preparar_dataset_hf(splits[tag]["treino"]),
        "val":    preparar_dataset_hf(splits[tag]["val"]),
        "teste":  preparar_dataset_hf(splits[tag]["teste"]),
    }
    print(f"{tag}: tokenização concluída "
          f"({len(datasets_tokenizados[tag]['treino'])} treino, "
          f"{len(datasets_tokenizados[tag]['val'])} val, "
          f"{len(datasets_tokenizados[tag]['teste'])} teste)")

# Checagem rápida: mostra um exemplo tokenizado para conferência visual
exemplo = datasets_tokenizados["80_10_10"]["treino"][0]
print("\nExemplo tokenizado:")
print("input_ids (primeiros 15):", exemplo["input_ids"][:15])
print("labels:", exemplo["labels"], "->", LABEL_NAMES[int(exemplo["labels"])])
print("weight:", exemplo["weight"])

# --- Checagem de taxa de truncamento em MAX_LENGTH=64 ---
def contar_truncados(df, tokenizer, max_length):
    truncados = 0
    for texto in df["relato"]:
        ids_completos = tokenizer(texto, truncation=False)["input_ids"]
        if len(ids_completos) > max_length:
            truncados += 1
    return truncados, len(df)

n_truncados, n_total = contar_truncados(splits["80_10_10"]["treino"], tokenizer, MAX_LENGTH)
print(f"\nRelatos truncados em MAX_LENGTH={MAX_LENGTH} (treino, JabuticaBERT): "
      f"{n_truncados}/{n_total} ({100*n_truncados/n_total:.1f}%)")

config.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.37MB            

spm.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Map:   0%|          | 0/2432 [00:00<?, ? examples/s]

... (7 outputs left out)

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

60_20_20: tokenização concluída (1824 treino, 608 val, 609 teste)

Exemplo tokenizado:
input_ids (primeiros 15): tensor([   2,  342,  268,  275,  292,  281,  278,  327, 1420,  267,  404,  329,
         398,  281,  828])
labels: tensor(0) -> Bateria/Sistema Elétrico
weight: tensor(0.5000)

Relatos truncados em MAX_LENGTH=64 (treino, JabuticaBERT): 8/2432 (0.3%)


## Passo 3 — `WeightedTrainer` e métricas

Idêntico aos notebooks anteriores.

In [3]:
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(reduction="none")
        per_example_loss = loss_fct(logits, labels)
        loss = (per_example_loss * weights).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

print("WeightedTrainer e compute_metrics definidos.")

WeightedTrainer e compute_metrics definidos.


## Passo 4 — Treino leve comparativo (3 configs)

In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

def rodar_treino_leve(tag, epocas=3):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS
    )

    args = TrainingArguments(
        output_dir=f"./resultados_leve_jabuticabert_{tag}",
        num_train_epochs=epocas,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        seed=SEED,
        remove_unused_columns=False,   # crítico — não remover
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets_tokenizados[tag]["treino"],
        eval_dataset=datasets_tokenizados[tag]["val"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics_val = trainer.evaluate()
    return metrics_val["eval_f1_macro"], metrics_val["eval_accuracy"]

resultados_leve = {}
for tag in CONFIGS:
    print(f"\n{'='*60}\nTreino leve — config {tag}\n{'='*60}")
    f1, acc = rodar_treino_leve(tag, epocas=3)
    resultados_leve[tag] = {"f1_macro": f1, "accuracy": acc}
    print(f"{tag}: F1-macro (val) = {f1:.4f} | acurácia (val) = {acc:.4f}")

print("\n=== Comparação final (treino leve, JabuticaBERT) ===")
for tag, m in sorted(resultados_leve.items(), key=lambda x: -x[1]["f1_macro"]):
    print(f"{tag}: F1-macro={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}")

# Lembrete: diferenças de F1 abaixo de ~0.01 devem ser tratadas como empate
# técnico nesta escala de dataset.


Treino leve — config 80_10_10


model.safetensors: reconstructing file:   0%|          |  0.00B /  283MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: amadeusai/JabuticaBERT-XSmall
Key                 | Status  | 
--------------------+---------+-
classifier.weight   | MISSING | 
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 
classifier.bias     | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.213474,1.956038,0.229740,0.308197
2,1.618896,1.328660,0.565949,0.586885
3,1.202698,1.130663,0.644062,0.665574


... (8 outputs left out)

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: amadeusai/JabuticaBERT-XSmall
Key                 | Status  | 
--------------------+---------+-
classifier.weight   | MISSING | 
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 
classifier.bias     | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.308461,2.288112,0.058895,0.111842
2,2.145681,1.944761,0.372248,0.429276
3,1.744700,1.715557,0.448445,0.493421


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
1.744700,1.715557,3,0.448445,0.493421


60_20_20: F1-macro (val) = 0.4484 | acurácia (val) = 0.4934

=== Comparação final (treino leve, JabuticaBERT) ===
80_10_10: F1-macro=0.6441  acc=0.6656
70_15_15: F1-macro=0.6234  acc=0.6414
60_20_20: F1-macro=0.4484  acc=0.4934


## Passo 5 — Escolha da configuração vencedora

In [5]:
# Ajustem esta linha de acordo com o resultado do Passo 4
CONFIG_VENCEDORA = "80_10_10"  # <-- editar se outra config vencer

print(f"Config escolhida para o treino final: {CONFIG_VENCEDORA}")
print(f"Treino: {len(datasets_tokenizados[CONFIG_VENCEDORA]['treino'])} exemplos")
print(f"Validação: {len(datasets_tokenizados[CONFIG_VENCEDORA]['val'])} exemplos")
print(f"Teste: {len(datasets_tokenizados[CONFIG_VENCEDORA]['teste'])} exemplos (fica intocado até a avaliação final)")

Config escolhida para o treino final: 80_10_10
Treino: 2432 exemplos
Validação: 305 exemplos
Teste: 304 exemplos (fica intocado até a avaliação final)


## Passo 6 — Treino completo com early stopping

Mesma estratégia dos notebooks anteriores: até 15 épocas, com early
stopping (patience=3) monitorando o F1-macro de validação.

In [6]:
from transformers import EarlyStoppingCallback

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS
)

args_final = TrainingArguments(
    output_dir="./resultados_final_jabuticabert",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    seed=SEED,
    remove_unused_columns=False,
    report_to="none",
)

trainer_final = WeightedTrainer(
    model=model_final,
    args=args_final,
    train_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["treino"],
    eval_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["val"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_final.train()

print("\nMelhor F1-macro de validação atingido:",
      trainer_final.state.best_metric)
print("Checkpoint carregado automaticamente (load_best_model_at_end=True).")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: amadeusai/JabuticaBERT-XSmall
Key                 | Status  | 
--------------------+---------+-
classifier.weight   | MISSING | 
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 
classifier.bias     | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.189471,1.668946,0.484619,0.508197
2,1.269905,0.873822,0.753358,0.760656
3,0.647088,0.745053,0.785114,0.790164
4,0.404980,0.579177,0.818953,0.819672
5,0.249700,0.551689,0.828665,0.829508
6,0.148703,0.669454,0.836282,0.836066
7,0.090509,0.702471,0.808816,0.813115
8,0.069010,0.733883,0.840683,0.842623
9,0.047402,0.741782,0.859662,0.859016
10,0.032705,0.738507,0.861648,0.862295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Melhor F1-macro de validação atingido: 0.8616476876955582
Checkpoint carregado automaticamente (load_best_model_at_end=True).


## Passo 7 — Avaliação final no conjunto de teste

In [7]:
resultado_teste = trainer_final.evaluate(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
print("=== Resultado final no TESTE (JabuticaBERT) ===")
print(f"F1-macro: {resultado_teste['eval_f1_macro']:.4f}")
print(f"Acurácia: {resultado_teste['eval_accuracy']:.4f}")

# Relatório por classe — essencial para comparar com os demais modelos,
# em especial no par Superaquecimento x Arrefecimento
preds_output = trainer_final.predict(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels_reais = preds_output.label_ids

print("\n=== Relatório por classe ===")
print(classification_report(labels_reais, preds, target_names=LABEL_NAMES, digits=3))

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.014600,0.968129,13,0.838683,0.842105


=== Resultado final no TESTE (JabuticaBERT) ===
F1-macro: 0.8387
Acurácia: 0.8421



=== Relatório por classe ===
                                precision    recall  f1-score   support

      Bateria/Sistema Elétrico      0.714     0.968     0.822        31
             Sistema de Freios      0.806     0.806     0.806        31
     Superaquecimento do Motor      0.810     0.567     0.667        30
       Suspensão/Amortecedores      0.778     0.677     0.724        31
 Sistema de Transmissão/Câmbio      0.968     0.968     0.968        31
             Vazamento de Óleo      0.967     0.967     0.967        30
      Sistema de Arrefecimento      0.800     0.933     0.862        30
                     Pneu/Roda      0.929     0.867     0.897        30
Sistema de Injeção/Combustível      0.815     0.733     0.772        30
        Sistema de Escapamento      0.875     0.933     0.903        30

                      accuracy                          0.842       304
                     macro avg      0.846     0.842     0.839       304
                  weighted avg  

## Passo 8 — Salvar o modelo

In [8]:
from google.colab import drive
drive.mount('/content/drive')

CAMINHO_SALVAR = '/content/drive/MyDrive/Agente Mecânico/modelos/jabuticabert_mao_na_roda/'

trainer_final.save_model(CAMINHO_SALVAR)
tokenizer.save_pretrained(CAMINHO_SALVAR)

print(f"Modelo e tokenizer salvos em: {CAMINHO_SALVAR}")
print("Conteúdo salvo:")
!ls -la "{CAMINHO_SALVAR}"

import json
resumo = {
    "modelo": MODEL_CHECKPOINT,
    "config_split": CONFIG_VENCEDORA,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "f1_macro_teste": float(resultado_teste["eval_f1_macro"]),
    "acuracia_teste": float(resultado_teste["eval_accuracy"]),
    "melhor_f1_validacao": float(trainer_final.state.best_metric),
}
with open(CAMINHO_SALVAR + "resumo_treino.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("\nResumo salvo em resumo_treino.json:")
print(json.dumps(resumo, ensure_ascii=False, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo e tokenizer salvos em: /content/drive/MyDrive/Agente Mecânico/modelos/jabuticabert_mao_na_roda/
Conteúdo salvo:
total 283459
drwx------ 2 root root      4096 Sep 16 00:06 .
drwx------ 7 root root      4096 Sep 16 00:05 ..
-rw------- 1 root root      1393 Sep 16 00:05 config.json
-rw------- 1 root root 283359760 Sep 16 00:06 model.safetensors
-rw------- 1 root root       602 Sep 16 00:06 tokenizer_config.json
-rw------- 1 root root   6884361 Sep 16 00:06 tokenizer.json
-rw------- 1 root root      5165 Sep 16 00:06 training_args.bin

Resumo salvo em resumo_treino.json:
{
  "modelo": "amadeusai/JabuticaBERT-XSmall",
  "config_split": "80_10_10",
  "seed": 42,
  "max_length": 64,
  "f1_macro_teste": 0.8386828437663499,
  "acuracia_teste": 0.8421052631578947,
  "melhor_f1_validacao": 0.8616476876955582
}
